# Common-budget experiment: source functions

This notebook contains the policy, deployment and common-budget evaluation
functions used by `07_common_budget_all_regimes.ipynb` and the three-arm control
wrapper. Use those entry notebooks to run the experiments.

The common-budget comparison sets the loss budget to 5 bp for both policy
arms. The v1.3 replay applies the paper's warm-up and batched-feedback
conventions to the resulting episode records.


## Environment


In [ ]:
# CTC local runtime
from pathlib import Path
import os, sys, json, time, math, random, glob, hashlib, importlib, shutil, copy
from itertools import product
from scipy import stats
from collections import deque
def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

KIT_ROOT = _find_repository_root()
if str(KIT_ROOT) not in sys.path:
    sys.path.insert(0, str(KIT_ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display
from ctc_runtime import initialize, generate_rollout

globals().update(initialize())
os.chdir(LOCAL_REPO)
DEVICE = torch.device('cpu')
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print('Repository:', KIT_ROOT)
print('Device:', DEVICE, '| torch:', torch.__version__, '| threads:', torch.get_num_threads())


## Train, validation and test years


In [ ]:
# ── REGIME config (mixed multi-year training version) ──
#  mixed training: TRAIN_YEARS can be"long and diverse"year list(including bull/bear/crash),
#  so g1 learns to control the tail across all regimes. But it must strictly satisfy: every train year < valyear < testyear.

# ── preset: uncomment one of these lines (crash panel: run first crash_test2018, then change to crash_test2020 rerun the fullnotebook) ──
REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2018', list(range(2008,2017)), [2017], [2018]    # clean OOS crash: Feb Volmageddon + Q4 sharp drop
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2020', list(range(2008,2019)), [2019], [2020]   # clean OOS crash: COVID (full year + crash slice)
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2022', list(range(2008,2021)), [2021], [2022]      # mixed 13 years → bear market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2024', list(range(2008,2023)), [2023], [2024]      # mixed → bull market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'narrow_test2022', [2019,2020], [2021], [2022]              # narrow-training control

# ── strictly guard against look-ahead (hard assertions): every train year < every val year < every test year ──
assert max(TRAIN_YEARS) < min(VAL_YEARS),  f'LOOK-AHEAD! train latest year {max(TRAIN_YEARS)} no earlier than val {min(VAL_YEARS)}'
assert max(VAL_YEARS)   < min(TEST_YEARS), f'LOOK-AHEAD! val latest year {max(VAL_YEARS)} no earlier than test {min(TEST_YEARS)}'
assert not (set(TRAIN_YEARS) | set(VAL_YEARS)) & set(TEST_YEARS), 'test year must not appear in train/val'

# ── crash sub-window: in"full year test", and within it separately evaluate the true crash period (verdict cell use) ──
#    decision rule: episode live interval [date_start, date_end] with any crash window"overlaps"counts it in,
#    because if the holding period touches the crash at all, this hedge has lived through the crash.
CRASH_WINDOWS = {
    'crash_test2018': [('2018-02-01', '2018-02-12'),   # Volmageddon (XIV blow-up, VIX doubled)
                       ('2018-10-01', '2018-12-26')],  # Q4 selloff all the way to the Christmas low
    'crash_test2020': [('2020-02-19', '2020-04-07')],  # COVID crash (top -> 3/23 bottom -> initial rebound)
}
CRASH_SLICES = CRASH_WINDOWS.get(REGIME, [])

OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{REGIME}'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(CLEAN_PARQUET), f'missing parquet: {CLEAN_PARQUET}'
print(f'REGIME={REGIME}')
print(f'  TRAIN={list(TRAIN_YEARS)}  (all < val)')
print(f'  VAL  ={VAL_YEARS}  (< test)')
print(f'  TEST ={TEST_YEARS}')
if CRASH_SLICES: print(f'  CRASH sub-slice={CRASH_SLICES}')
print('✓ strictly forward: max(train) < min(val) <= max(val) < min(test) — no look-ahead')


## Construct the data splits


In [ ]:
# ── split years by REGIME train/val/test ──
df_all = pd.read_parquet(CLEAN_PARQUET)
df_all['date'] = pd.to_datetime(df_all['date'])
yr = df_all['date'].dt.year
OUT_COLS = ['quote_datetime','ticker','expiration','strike',
            'underlying_bid','underlying_ask','bid','ask','nbr_next_steps']
for name, years in [('train', TRAIN_YEARS), ('validation', VAL_YEARS), ('test', TEST_YEARS)]:
    sub = df_all.loc[yr.isin(years)].sort_values(['expiration','strike','date']).reset_index(drop=True)
    g = sub.groupby(['expiration','strike'], sort=False)
    sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
    sub[OUT_COLS].to_csv(f'data/{name}.csv', index=False)
    print(f'  {name:11s}: years={years}  rows={len(sub):>9,}')
rates = (df_all[['date','r']].drop_duplicates('date').sort_values('date')
         .rename(columns={'date':'Date','r':'1y'}))
rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
rates = pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date')
rates.to_csv('data/1yr_treasury.csv', index=False)
print('treasury OK')


## Black–Scholes Greeks


In [ ]:
from scipy.stats import norm

def bs_greeks_inner(S, K, r, q, vol, T_years):
    if T_years <= 0 or vol <= 1e-8 or S <= 0 or K <= 0:
        return 0.0, 0.0, 0.0, 0.0
    sqT = np.sqrt(T_years)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol) * T_years) / (vol * sqT)
    d2 = d1 - vol * sqT
    n_d1 = norm.pdf(d1)
    return (float(norm.cdf(d1)),
            float(n_d1 / (S * vol * sqT)),
            float(S * n_d1 * sqT),
            float(-(S * n_d1 * vol) / (2 * sqT) - r * K * np.exp(-r * T_years) * norm.cdf(d2)))


## Evaluation functions


In [ ]:
STATE_DIM, ACTION_DIM = 8, 1
TC_RATE = 0.001
REWARD_EXP = 1.0
GAMMA = 0.99
Q_DIV = 0.0
LAMBDAS_DEC = [0.0, 0.25, 0.5, 0.75, 1.0]
RATES = pd.read_csv('data/1yr_treasury.csv')
RATES['Date'] = pd.to_datetime(RATES['Date'])

def get_r(date):
    rows = RATES[RATES['Date']==date]
    if len(rows): return float(rows['1y'].iloc[0])
    rows = RATES[RATES['Date']<=date].sort_values('Date')
    return float(rows['1y'].iloc[-1]) if len(rows) else 0.04

def build_state(row, position_before_action):
    T_days = float(row['T'])
    spot   = float(row['S-1'])
    K      = spot / float(row['S/K'])
    IV     = float(row['v'])
    date   = pd.to_datetime(row['Date'])
    r      = get_r(date)
    d, g, vega, theta = bs_greeks_inner(spot, K, r, Q_DIV, IV, max(T_days, 0)/365.0)
    return np.array([float(row['S/K']), T_days/30.0, position_before_action, IV,
                     d, g, vega, theta], dtype=np.float32)

def cvar_5(arr):
    arr = np.asarray(arr); n_tail = max(1, int(0.05 * len(arr)))
    return np.sort(arr)[:n_tail].mean()


## Generate episode paths


In [ ]:
SEARCH_KAPPA = 2
REP_NAME = 'bs_reference'

def _csv_years(path):
    return set(pd.to_datetime(pd.read_csv(path, usecols=['Date'])['Date']).dt.year.astype(int))

def _file_md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()[:12]

def gen_rollout(split):
    return generate_rollout(split, REGIME, LOCAL_REPO)

ROLLF = {split: gen_rollout(split) for split in ('train', 'validation', 'test')}

def _episodes_from_csv(p):
    csv = pd.read_csv(p); csv['Date'] = pd.to_datetime(csv['Date'])
    eps = []
    for eid, edf in csv.groupby('episode', sort=False):
        edf = edf.sort_values('gstep').reset_index(drop=True)
        eps.append({'episode': int(eid), 'date_start': edf['Date'].iloc[0], 'ep_step_df': edf})
    return pd.DataFrame(eps).sort_values('date_start').reset_index(drop=True)

def load_eps(split): return _episodes_from_csv(ROLLF[split])

for split in ROLLF:
    print(f'{split}: {len(load_eps(split))} episodes')


## Residual policy and mean–CVaR objective


In [ ]:
# ── differentiable episode replay + BS residual policy + Rockafellar–Uryasev CVaR training ──
import torch, torch.nn as nn
DEVICE = torch.device('cpu')

def pack_episodes(eps_df):
    """flatten several episodes into flat tensors, for differentiable replay (state use the BS position trajectory, decoupled from the policy)."""
    states, b_pos, S_old, S_new, P_old, P_new, ep_id, is_first = [], [], [], [], [], [], [], []
    for ei, (_, erow) in enumerate(eps_df.iterrows()):
        df = erow['ep_step_df']; prev_b = 0.0
        for t in range(len(df)):
            r = df.iloc[t]
            states.append(build_state(r, prev_b))
            b = float(r['B Pos'])
            b_pos.append(b); S_old.append(float(r['S-1'])); S_new.append(float(r['S0']))
            P_old.append(float(r['P-1'])); P_new.append(float(r['P0']))
            ep_id.append(ei); is_first.append(1.0 if t == 0 else 0.0)
            prev_b = b
    t = lambda x, dt=torch.float32: torch.tensor(np.array(x), dtype=dt, device=DEVICE)
    return dict(states=t(states), b_pos=t(b_pos), S_old=t(S_old), S_new=t(S_new),
                P_old=t(P_old), P_new=t(P_new), ep_id=t(ep_id, torch.long),
                is_first=t(is_first), n_ep=int(eps_df.shape[0]))

class ResidualPolicy(nn.Module):
    def __init__(self, dim=8, hid=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid), nn.ReLU(),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)  # initial≈BS
    def forward(self, s): return self.net(s).squeeze(-1)

def _seg_sum(vals, ep_id, n_ep):
    out = torch.zeros(n_ep, device=vals.device, dtype=vals.dtype)
    return out.index_add_(0, ep_id, vals)

def _episode_reward(pack, pos, kappa):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    q = pnl * 100.0
    step_r = (q - kappa * torch.relu(-q)) * 10.0          # new downside reward, differentiable (relu)
    return _seg_sum(step_r, pack['ep_id'], pack['n_ep'])

def _positions(pack, policy):
    adj = policy(pack['states'])
    return torch.clamp(pack['b_pos'] + adj, -1.5, 1.5)     # position = BS delta + residual (B Pos=-delta∈[-1,0])

def _train_residual_once(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0,
                         init_std=0.01, warm_frac=0.25):
    """single training run: small random init of the last layer + pure mean for the first warm_frac(β=0) + β ramp linearly to the target (v9)."""
    torch.manual_seed(seed); np.random.seed(seed)
    policy = ResidualPolicy().to(DEVICE)
    torch.nn.init.normal_(policy.net[-1].weight, std=init_std)
    torch.nn.init.zeros_(policy.net[-1].bias)
    eta = torch.zeros(1, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam(list(policy.parameters()) + [eta], lr=lr)
    bs_r = _episode_reward(pack_tr, pack_tr['b_pos'], kappa).detach()
    for ep in range(epochs):
        bt = 0.0 if ep < warm_frac * epochs else \
             beta * min(1.0, (ep - warm_frac*epochs) / max(1.0, warm_frac*epochs))
        opt.zero_grad()
        delta = _episode_reward(pack_tr, _positions(pack_tr, policy), kappa) - bs_r
        cvar = eta - (1.0 / alpha) * torch.relu(eta - delta).mean()
        loss = -(delta.mean() + bt * cvar)
        loss.backward(); opt.step()
    return policy

def _adj_scale(policy, pack):
    with torch.no_grad():
        return policy(pack['states']).abs().mean().item()

COLLAPSE_TH, N_RETRY = 1e-3, 1   # collapse criterion(trainside) and restart count — report the trigger frequency in the paper's methods section

def train_residual(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0, **kw):
    """v9 = warmup training + auto-restart on collapse (the criterion looks only at the train side |adj|, does not touch test; seed+1000 retry)."""
    policy = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed, **kw)
    a = _adj_scale(policy, pack_tr); tries = 0
    while a < COLLAPSE_TH and tries < N_RETRY:
        tries += 1
        print(f'  ↻ collapse-retry{tries}: κ={kappa} β={beta} seed {seed}→{seed+1000*tries} (|adj|={a:.5f})')
        p2 = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed + 1000*tries, **kw)
        a2 = _adj_scale(p2, pack_tr)
        if a2 > a: policy, a = p2, a2
    if a < COLLAPSE_TH:
        print(f'  ⚠ collapse persists: κ={kappa} β={beta} seed={seed} (|adj|={a:.5f})')
    return policy

@torch.no_grad()
def eval_delta(pack, policy, kappa):
    bs_r = _episode_reward(pack, pack['b_pos'], kappa)
    return (_episode_reward(pack, _positions(pack, policy), kappa) - bs_r).cpu().numpy()

@torch.no_grad()
def eval_rewards(pack, policy, kappa):
    r_pol = _episode_reward(pack, _positions(pack, policy), kappa).cpu().numpy()
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa).cpu().numpy()
    return r_pol, r_bs

print('residual-CVaR machinery ready; device =', DEVICE)


## Training diagnostics


In [ ]:
import hashlib as _ahl, inspect as _ains
def _audit(tag, pack=None):
    parts = [f'[audit:{tag}]']
    try:
        parts.append('RATES=' + _ahl.md5(open('data/1yr_treasury.csv','rb').read()).hexdigest()[:8]
                     + f'({len(RATES)})')
    except Exception: parts.append('RATES=?')
    for f in (train_residual, _train_residual_once, _episode_reward, _positions, build_state):
        parts.append(f.__name__[:6] + '=' + _ahl.md5(_ains.getsource(f).encode()).hexdigest()[:6])
    if pack is not None:
        parts.append('states=' + _ahl.md5(pack['states'].detach().cpu().numpy().tobytes()).hexdigest()[:10])
    import torch as _t
    parts.append(f'thr={_t.get_num_threads()}')
    print(' '.join(parts))
print('audit ready')

## Paired estimates and block bootstrap


In [ ]:
import numpy as np

def _ep_dates(eps_df):
    """each episode's start date (datetime64, pack order)."""
    return pd.to_datetime(eps_df['date_start']).dt.floor('D').to_numpy()

def cvar5(a):
    a = np.asarray(a, float); k = max(1, int(0.05 * len(a)))
    return float(np.sort(a)[:k].mean())

def full_stat_ci(x, dates, stat=np.mean, B=3000, block_days=10, seed=0):
    """main convention: full-sample statistics + date-based blocks bootstrap CI.
    resampling unit=trading-day blocks (same-day episodes enter/exit as a whole), block length block_days absorbs temporal correlation."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    groups = [np.where(inv == k)[0] for k in range(len(d_uniq))]
    rng = np.random.default_rng(seed)
    point = float(stat(x))
    n = len(d_uniq); nb = int(np.ceil(n / block_days))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        di = np.concatenate([(s + np.arange(block_days)) % n for s in starts])[:n]
        sel = np.concatenate([groups[k] for k in di])
        bs[b] = stat(x[sel])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return point, float(lo), float(hi), float((bs <= 0).mean())

def collapsed_spaced(x, dates, min_days=7):
    """auxiliary convention: same-day mean collapse → ≥min_days greedy spacing. Deterministic, no ties."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    daily = np.array([x[inv == k].mean() for k in range(len(d_uniq))])
    sel, last = [], None
    for k in range(len(d_uniq)):
        if last is None or (d_uniq[k] - last) / np.timedelta64(1, 'D') >= min_days:
            sel.append(k); last = d_uniq[k]
    sel = np.array(sel, dtype=int)
    return d_uniq[sel], daily[sel]

def spaced_mean_ci(x, dates, min_days=7, B=3000, L=5, seed=0):
    _, v = collapsed_spaced(x, dates, min_days)
    rng = np.random.default_rng(seed)
    n = len(v); nb = int(np.ceil(n / L))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        idx = np.concatenate([(s + np.arange(L)) % n for s in starts])[:n]
        bs[b] = v[idx].mean()
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(v.mean()), float(lo), float(hi), float((bs <= 0).mean()), n

def summarize_dp(dp, dates, tag=''):
    """one-line aggregate: main-convention trio + auxiliary convention. Returns a dict for the summary table."""
    m, lo, hi, p = full_stat_ci(dp, dates, np.mean)
    w, wl, wh, _ = full_stat_ci(dp, dates, lambda a: float((np.asarray(a) > 0).mean()))
    t, tl, th, _ = full_stat_ci(dp, dates, cvar5)
    sm, sl, sh, sp_, nsp = spaced_mean_ci(dp, dates)
    print(f'{tag} full sample: ΔPnL={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] p_le0={p:.3f} | '
          f'win={w:.0%} [{wl:.0%},{wh:.0%}] | tailCVaR={t:+.2f} [{tl:+.2f},{th:+.2f}]')
    print(f'{tag} collapse by day×≥7days (n={nsp}): ΔPnL={sm:+.2f}bp [{sl:+.2f},{sh:+.2f}] p_le0={sp_:.3f}')
    return dict(full=m, full_lo=lo, full_hi=hi, full_p=p, win=w, win_lo=wl, win_hi=wh,
                tail=t, tail_lo=tl, tail_hi=th, sp=sm, sp_lo=sl, sp_hi=sh, sp_p=sp_,
                n_spaced=nsp)

print('✓ E0 unified estimator ready (full_stat_ci / collapsed_spaced / summarize_dp)')

## Regime configuration


In [ ]:
# Switch the data split and load its validation-selected beta.
_REGIME_PRESETS = {
    'crash_test2018': (list(range(2008, 2017)), [2017], [2018]),
    'crash_test2020': (list(range(2008, 2019)), [2019], [2020]),
    'mix_test2022':   (list(range(2008, 2021)), [2021], [2022]),
    'mix_test2024':   (list(range(2008, 2023)), [2023], [2024]),
}

def switch_regime(regime):
    """switch the global context to the specified regime: years/split/cache incorporation/ROLLF/fingerprint validation/β*load."""
    global REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS, CRASH_SLICES, OUT_DIR, ROLLF
    global CFG_TAG, _SPLIT_YEARS, RATES, best_beta
    global ALPHA, EPOCHS, LR, KAPPAS, POLICY_SEEDS
    import hashlib, glob, shutil, json as _json

    ty, vy, te_y = _REGIME_PRESETS[regime]
    assert max(ty) < min(vy) and max(vy) < min(te_y), 'LOOK-AHEAD!'
    REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = regime, ty, vy, te_y
    CRASH_SLICES = CRASH_WINDOWS.get(regime, [])
    OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{regime}'
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f'━━ switch → {regime}: train {ty[0]}-{ty[-1]} | val {vy} | test {te_y} '
          f'| crashslice={bool(CRASH_SLICES)}')

    # 1) rewrite data/{train,validation,test}.csv + Treasury rate (and ## 2 same logic)
    df_all = pd.read_parquet(CLEAN_PARQUET)
    df_all['date'] = pd.to_datetime(df_all['date'])
    yr = df_all['date'].dt.year
    out_cols = ['quote_datetime', 'ticker', 'expiration', 'strike',
                'underlying_bid', 'underlying_ask', 'bid', 'ask', 'nbr_next_steps']
    for name, years in [('train', ty), ('validation', vy), ('test', te_y)]:
        sub = df_all.loc[yr.isin(years)].sort_values(
            ['expiration', 'strike', 'date']).reset_index(drop=True)
        g = sub.groupby(['expiration', 'strike'], sort=False)
        sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
        sub[out_cols].to_csv(f'data/{name}.csv', index=False)
    rates = (df_all[['date', 'r']].drop_duplicates('date').sort_values('date')
             .rename(columns={'date': 'Date', 'r': '1y'}))
    rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
    old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
    pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date').to_csv(
        'data/1yr_treasury.csv', index=False)
    RATES = pd.read_csv('data/1yr_treasury.csv'); RATES['Date'] = pd.to_datetime(RATES['Date'])
    del df_all

    # 2) recomputed for this regime CFG_TAG / _SPLIT_YEARS (fixes the batch-phase tag freeze issue)
    CFG_TAG = hashlib.md5(repr((sorted(TRAIN_YEARS), sorted(VAL_YEARS), sorted(TEST_YEARS),
                                REP_NAME)).encode()).hexdigest()[:8]
    _SPLIT_YEARS = {'train': set(map(int, TRAIN_YEARS)),
                    'validation': set(map(int, VAL_YEARS)),
                    'test': set(map(int, TEST_YEARS))}

    # 4) ROLLF + hard validation + fingerprint archived
    ROLLF = {sp: gen_rollout(sp) for sp in ['train', 'validation', 'test']}
    import torch as _th
    print(f'[env] torch={_th.__version__}  '
          f'dev={_th.cuda.get_device_name(0) if _th.cuda.is_available() else "CPU"}')
    print(f'[fp ] parquet md5={_file_md5(CLEAN_PARQUET)}')
    for sp in ['train', 'validation', 'test']:
        yrs = _csv_years(ROLLF[sp]); exp = _SPLIT_YEARS[sp]
        assert yrs <= exp, f'{sp}: year out of range {sorted(yrs - exp)}!'
        assert not (exp - yrs), f'{sp}: missing years {sorted(exp - yrs)}!'
        print(f'[fp ] {sp:11s} {min(yrs)}–{max(yrs)} ✓  md5={_file_md5(ROLLF[sp])}')

    # 5) constants + β* (reads the batch-phase on-disk json, does not rerun the search)
    ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
    KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
    with open(f'{OUT_DIR}/regime_best_beta.json') as f:
        best_beta = {int(k): float(v) for k, v in _json.load(f).items()}
    print(f'★ β* = {best_beta}   (from {OUT_DIR}/regime_best_beta.json)')
    print('Regime ready:', REGIME)

print('switch_regime() ready — usage: switch_regime("mix_test2022")')


## Four-regime deployment comparison

Set `PANEL_REGIME` in H0 and run H0–H5 for that regime. Restart the kernel
before changing regimes. Once all four runs finish, execute H6 to combine
the saved results.

For 2022, the comparison checks the saved six-rule statistics. For the other
regimes, repeated training with the same seed checks deterministic replay.


### H0. Select a regime and load inputs


In [ ]:
# ── H0. PANEL selector: uncomment one of the four lines; changing regime requires restarting the kernel and rerunning the whole notebook ──
#PANEL_REGIME = 'crash_test2018'
#PANEL_REGIME = 'crash_test2020'
#PANEL_REGIME = 'mix_test2022'     # the only one with v3 six-rule/two-arm sealed artifacts → can run in full GATE2/GATE3
PANEL_REGIME = 'mix_test2024'

# mild arm β=0.5: only needed for 2022(replicate GATE3). By default the other regimes run only the wild arm β=0 (saves half the compute);
# if you want to see β=0.5 of ε frontier, you can manually set True.
DO_MILD = (PANEL_REGIME == 'mix_test2022')

switch_regime(PANEL_REGIME)
ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
EPS_TR, EPS_VA, EPS_TE = load_eps('train'), load_eps('validation'), load_eps('test')
pack_tr, pack_va, pack_te = (pack_episodes(EPS_TR), pack_episodes(EPS_VA), pack_episodes(EPS_TE))
_audit('H:prep', pack_tr)
BEST_BETA_BASE = dict(best_beta)
DATES_TE = _ep_dates(EPS_TE)
SP_TE = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_TE.iterrows()])

def _episode_pnl(pack, pos):                      # and v3 PART B/C same definition
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])
BS_PNL_TE = _episode_pnl(pack_te, pack_te['b_pos']).detach().cpu().numpy()

UNI_DIR = f'{DRIVE_REPO}/checkpoints/unified_v3'
V3_DIR  = f'{DRIVE_REPO}/checkpoints/conformal_v3_mix_test2022'
for k in KAPPAS:                                  # GATE1 reference (must be in place for all regimes)
    _f = f'{UNI_DIR}/dp_{PANEL_REGIME}_k{k}.npy'
    assert os.path.exists(_f), f'missing v3 panel sealed artifacts: {_f} — first fully run python scripts/run_experiments.py --stage layer1'
HAS_V3_SIXARMS = (PANEL_REGIME == 'mix_test2022'
                  and os.path.exists(f'{V3_DIR}/six_rules_mix_test2022.csv')
                  and os.path.exists(f'{V3_DIR}/arms_mix_test2022.pkl'))
print(f'H0 ready | regime={PANEL_REGIME} | β*={BEST_BETA_BASE} | DO_MILD={DO_MILD}')
print(f'  cross-session GATE2/3 reference = {"in place(2022)" if HAS_V3_SIXARMS else "none → this regime uses the same-session determinism gate"}')


### Deployment rules


In [ ]:
# ── G2. extended CRC mechanism: v3 closed-loop mechanism + four-control rules (fixv / valstatic / emp / ivrule) ──
EXP_NAME = 'conformal_controls'
EXP_DIR  = f'{DRIVE_REPO}/checkpoints/{EXP_NAME}_{REGIME}'; os.makedirs(EXP_DIR, exist_ok=True)
LAM_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
EPS_T    = 0.10

def _episode_pnl(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])

@torch.no_grad()
def episode_delta_lambda(pack, policy, kappa, lam_vec):
    """reward convention Δ (for evaluation, consistent with the v7 report)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    r_pol = _episode_reward(pack, pos, kappa)
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa)
    return (r_pol - r_bs).cpu().numpy(), r_pol.cpu().numpy(), r_bs.cpu().numpy()

@torch.no_grad()
def episode_dpnl_bp_lambda(pack, policy, lam_vec, start_spot):
    """PnL convention Δ, in bp of notional (for control/budget, κ-unrelated)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    dp = (_episode_pnl(pack, pos) - _episode_pnl(pack, pack['b_pos'])).cpu().numpy()
    return dp / start_spot * 1e4

def conformal_lower_q(v, w, eps):
    """weighted conformal lower tail ε quantile, virtual point −∞ weight 1 (conservative correction). Insufficient samples → −inf."""
    v = np.asarray(v, float); w = np.asarray(w, float)
    W = w.sum() + 1.0
    if W <= 0 or 1.0 / W >= eps: return -np.inf
    idx = np.argsort(v); v, w = v[idx], w[idx]
    cw = (1.0 + np.cumsum(w)) / W          # −∞ mass entirely at the left end
    k = int(np.searchsorted(cw, eps, side='left'))
    return float(v[min(k, len(v) - 1)])

def precompute_grids(eps_df, pack, policy, kappa):
    """each (policy, split) computed only once: date/spot/IV + all λ the grid's reward-Δ and PnL-Δbp."""
    n  = len(eps_df)
    ds = pd.to_datetime(eps_df['date_start']).to_numpy()
    de = np.array([pd.to_datetime(e['ep_step_df']['Date']).max()
                   for _, e in eps_df.iterrows()], dtype='datetime64[ns]')
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_df.iterrows()])
    iv = np.array([float(e['ep_step_df']['v'].iloc[0])  for _, e in eps_df.iterrows()])
    order = np.argsort(ds)
    d_lam  = {la: episode_delta_lambda(pack, policy, kappa, np.full(n, la))[0] for la in LAM_GRID}
    dp_lam = {la: episode_dpnl_bp_lambda(pack, policy, np.full(n, la), ss)     for la in LAM_GRID}
    return dict(n=n, ds=ds, de=de, ss=ss, iv=iv, order=order, d_lam=d_lam, dp_lam=dp_lam)

def calibrate_budget_bp(pack_va, eps_va_df, policy, eps_target, floor_bp=5.0):
    """budget c = validation fix1's ΔPnL_bp of ε absolute quantile value (lower bound floor_bp)."""
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_va_df.iterrows()])
    dp = episode_dpnl_bp_lambda(pack_va, policy, np.ones(len(eps_va_df)), ss)
    return float(max(floor_bp, -np.quantile(dp, eps_target)))

def deploy_lambda_control(eps_df, pack, policy, kappa, mode='crc_rwc', eps_target=0.10,
                          budget_bp=25.0, grids=None, gamma_eps=0.02, half_life=30.0,
                          tau_iv=0.04, warmup=15, M_emp=0.3, lam_warmup=1.0, lam_fix=None):
    """chronological episode-by-episode deployment λ_t.
       mode ∈ {fix1, fix0, fixv(λ=lam_fix), a2margin, emp, ivrule, aci_open, crc, crc_rwc, crc_rwc_aci}"""
    g = grids if grids is not None else precompute_grids(eps_df, pack, policy, kappa)
    n, ds, de, iv, order = g['n'], g['ds'], g['de'], g['iv'], g['order']
    d_lam, dp_lam = g['d_lam'], g['dp_lam']
    used = np.full(n, np.nan); informed = np.zeros(n, bool)
    lam_aci = float(lam_warmup); eps_eff = float(eps_target)
    aci_pending = []                                   # crc_rwc_aci delayed-feedback queue
    for t, i in enumerate(order):
        past = order[:t]
        done = past[de[past] < ds[i]]                  # ★ use only completed episode (fix look-ahead)
        informed[i] = len(done) >= warmup
        # —— crc_rwc_aci: first digest"completed by now"deployment results, online-tune ε_eff (closed-loop + delayed feedback) ——
        if mode == 'crc_rwc_aci' and aci_pending:
            still = []
            for j in aci_pending:
                if de[j] < ds[i]:
                    err = float(dp_lam[used[j]][j] < -budget_bp)
                    eps_eff = float(np.clip(eps_eff + gamma_eps * (eps_target - err),
                                            0.25 * eps_target, min(0.45, 4 * eps_target)))
                else:
                    still.append(j)
            aci_pending = still
        # —— decision ——
        if   mode == 'fix1': lam_t = 1.0
        elif mode == 'fix0': lam_t = 0.0
        elif mode == 'fixv': lam_t = float(lam_fix)    # controls one/two: any fixed λ (with valstatic)
        elif not informed[i]:
            lam_t = 0.0 if mode == 'a2margin' else lam_warmup
        elif mode == 'a2margin':                       # old A2-style (completion filtering fixed)
            thr = cvar_5(d_lam[0.0][done]) - M_emp
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if cvar_5(d_lam[la][done]) >= thr: lam_t = la; break
        elif mode == 'emp':                            # control three: rolling empirical breach-rate rule (no weighting/no correction)
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if float(np.mean(dp_lam[la][done] < -budget_bp)) <= eps_target:
                    lam_t = la; break
        elif mode == 'ivrule':                         # control four: IV quantile haircut (thresholds 50/80 preregistered)
            p_iv = float(np.mean(iv[done] <= iv[i]))
            lam_t = 1.0 if p_iv < 0.5 else (0.5 if p_iv < 0.8 else 0.25)
        elif mode == 'aci_open':                       # old open-loop rule (ablation control, only fixes completion filtering)
            var_hat = np.quantile(dp_lam[1.0][done], eps_target)
            err = float(dp_lam[1.0][i] < var_hat)      # events remain aligned with deployment λ unrelated → open-loop
            lam_aci = float(np.clip(lam_aci + 0.10 * (eps_target - err), 0.0, 1.0))
            lam_t = lam_aci
        else:                                          # crc / crc_rwc / crc_rwc_aci (closed-loop)
            if mode in ('crc_rwc', 'crc_rwc_aci'):
                dt_ = (ds[i] - de[done]) / np.timedelta64(1, 'D')
                w = (np.exp(-np.log(2) * np.maximum(dt_, 0.0) / half_life)
                     * np.exp(-np.abs(iv[done] - iv[i]) / tau_iv))
            else:
                w = np.ones(len(done))
            e_use = eps_eff if mode == 'crc_rwc_aci' else eps_target
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):  # risk varies with λ monotonically ⇒ take the largest feasible λ
                if conformal_lower_q(dp_lam[la][done], w, e_use) >= -budget_bp:
                    lam_t = la; break
        used[i] = lam_t
        if mode == 'crc_rwc_aci': aci_pending.append(i)
    # —— unified evaluation: reward basis + post-deployment actual breach (all modes comparable) ——
    d_used, rp, rbs = episode_delta_lambda(pack, policy, kappa, used)
    dp_used = episode_dpnl_bp_lambda(pack, policy, used, g['ss'])
    m = informed
    breach = float(np.mean(dp_used[m] < -budget_bp)) if m.any() else np.nan
    return {'lam': used, 'd': d_used, 'rp': rp, 'rbs': rbs, 'dpnl_bp': dp_used,
            'breach': breach, 'informed': m, 'budget_bp': float(budget_bp),
            'eps_eff_final': float(eps_eff)}

def lam_val_static(policy, kappa, budget_bp, eps_target=EPS_T):
    """control two: validation year, take max{λ: experience breach ≤ ε} (λ=0 always feasible ⇒ always has a solution)."""
    gva = precompute_grids(EPS_VA, pack_va, policy, kappa)
    for la in sorted(LAM_GRID, reverse=True):
        if float(np.mean(gva['dp_lam'][la] < -budget_bp)) <= eps_target:
            return float(la)
    return 0.0

def deploy_mode(mode, pol, kappa, sd, grids, lamval, budget):
    """unified entry point: fix25/50/75 and valstatic go through fixv, the rest native mode."""
    if mode in ('fix25', 'fix50', 'fix75'):
        return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode='fixv',
                                     lam_fix=float(mode[3:]) / 100.0, eps_target=EPS_T,
                                     budget_bp=budget, grids=grids)
    if mode == 'valstatic':
        return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode='fixv',
                                     lam_fix=lamval[(kappa, sd)], eps_target=EPS_T,
                                     budget_bp=budget, grids=grids)
    return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode=mode,
                                 eps_target=EPS_T, budget_bp=budget, grids=grids)

ALL_MODES = ['fix0', 'fix25', 'fix50', 'fix75', 'fix1', 'valstatic', 'ivrule', 'emp',
             'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
V3_SIX    = ['fix1', 'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
V3_ARM5   = ['fix1', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']

import hashlib as _h, inspect as _i
for _fn in [deploy_lambda_control, conformal_lower_q, precompute_grids,
            calibrate_budget_bp, lam_val_static, deploy_mode]:
    print('[src]', f'{_fn.__name__:22s}', _h.md5(_i.getsource(_fn).encode()).hexdigest()[:8])
print('extended CRC mechanism ready — 13 rule | EXP_DIR =', EXP_DIR)


### H1. Reproduce validation-selected policy rewards


In [ ]:
# ── H1. GATE1: base arm(β*) 15 policy retraining + comparison with v3 PART B dp_<regime> bitwise reconciliation ──
BASE = {}; _g1_ok = True
for kappa in KAPPAS:
    rows, dps = [], []
    for sd in POLICY_SEEDS:
        pol = train_residual(pack_tr, kappa, BEST_BETA_BASE[kappa], ALPHA, EPOCHS, LR, seed=sd)
        rows.append({'seed': sd, 'pol': pol})
        dps.append((_episode_pnl(pack_te, _positions(pack_te, pol)).detach().cpu().numpy()
                    - BS_PNL_TE) / SP_TE * 1e4)
    BASE[kappa] = rows
    dpm = np.mean(dps, axis=0)
    ref = np.load(f'{UNI_DIR}/dp_{PANEL_REGIME}_k{kappa}.npy')
    ok = bool(np.array_equal(dpm, ref)); _g1_ok &= ok
    print(f'κ={kappa} dp and unified_v3/dp_{PANEL_REGIME} bitwise identical: {ok}')
assert _g1_ok, 'GATE1 failed: retraining inconsistent with the v3 lineage — check the environment cell/threads=1/whether it was fully run ## 0–## 6'
GATE1 = True
print(f'★ GATE1 PASS [{PANEL_REGIME}] — the control group of this regime shares lineage with the v3 main table')


### H2. Evaluate the validation-selected policy


In [ ]:
# ── H2. base arm 13 rules (deployment is evaluation-only, zero retraining) + GATE2 adaptive ──
budgets = {(k, sd): calibrate_budget_bp(pack_va, EPS_VA, BASE[k][i]['pol'], EPS_T)
           for k in KAPPAS for i, sd in enumerate(POLICY_SEEDS)}
LAMVAL_BASE = {}
for kappa in KAPPAS:
    for i, sd in enumerate(POLICY_SEEDS):
        LAMVAL_BASE[(kappa, sd)] = lam_val_static(BASE[kappa][i]['pol'], kappa, budgets[(kappa, sd)])
print('valstatic λ_val (base arm, by κ take the mean across seeds):',
      {k: round(np.mean([LAMVAL_BASE[(k, s)] for s in POLICY_SEEDS]), 3) for k in KAPPAS})

ROWS, DPM, LPATH = [], {}, {}
for kappa in KAPPAS:
    grids_per_seed = {sd: precompute_grids(EPS_TE, pack_te, BASE[kappa][i]['pol'], kappa)
                      for i, sd in enumerate(POLICY_SEEDS)}
    for mode in ALL_MODES:
        paths, br, lam, lpa = [], [], [], []
        for i, sd in enumerate(POLICY_SEEDS):
            out = deploy_mode(mode, BASE[kappa][i]['pol'], kappa, sd,
                              grids_per_seed[sd], LAMVAL_BASE, budgets[(kappa, sd)])
            paths.append(out['dpnl_bp']); br.append(out['breach'])
            lam.append(float(np.nanmean(out['lam']))); lpa.append(out['lam'])
        dpm = np.mean(paths, axis=0)
        DPM[('base', kappa, mode)] = dpm
        LPATH[('base', kappa, mode)] = np.mean(np.vstack(lpa), axis=0)
        m, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
        t, tl, th, _ = full_stat_ci(dpm, DATES_TE, cvar5)
        ROWS.append({'arm': 'base', 'kappa': kappa, 'mode': mode, 'full': m, 'lo': lo, 'hi': hi,
                     'p_le0': p, 'breach': float(np.nanmean(br)), 'lam': float(np.mean(lam)),
                     'tail': t, 'tail_lo': tl, 'tail_hi': th})
        print(f'[base] κ={kappa} {mode:12s}: ΔPnL={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] '
              f'breach={np.nanmean(br):.4f} λ̄={np.mean(lam):.3f} tail={t:+.2f}')
    f1 = DPM[('base', kappa, 'fix1')].mean()
    for mode in ALL_MODES:
        if mode == 'fix1' or abs(f1) < 1e-12: continue
        print(f'   retention {mode:12s} = {DPM[("base", kappa, mode)].mean() / f1 * 100:.0f}%')
    print()
CTRL_BASE = pd.DataFrame(ROWS)

# —— GATE2 adaptive ——
if HAS_V3_SIXARMS:
    six_ref = pd.read_csv(f'{V3_DIR}/six_rules_mix_test2022.csv')
    GATE2_MAXD = 0.0
    for _, r in six_ref.iterrows():
        current_result = CTRL_BASE[(CTRL_BASE['kappa'] == r['kappa']) & (CTRL_BASE['mode'] == r['mode'])].iloc[0]
        for col in ['full', 'lo', 'hi', 'p_le0', 'breach', 'lam']:
            GATE2_MAXD = max(GATE2_MAXD, abs(float(current_result[col]) - float(r[col])))
    assert GATE2_MAXD < 1e-9, f'GATE2 failed: six-rule reproduction maxdiff={GATE2_MAXD:.3e}'
    GATE2_KIND = 'cross_session_vs_v3'
    print(f'★ GATE2 PASS — six-rule statistic-column reproduction maxdiff={GATE2_MAXD:.3e}')
else:
    import hashlib as _h2
    def _phash(pol):
        return _h2.md5(b''.join(p.detach().cpu().numpy().tobytes() for p in pol.parameters())).hexdigest()[:12]
    _hA = _phash(BASE[2][0]['pol'])
    _hB = _phash(train_residual(pack_tr, 2, BEST_BETA_BASE[2], ALPHA, EPOCHS, LR, seed=0))
    assert _hA == _hB, f'GATE2(determinism) failed: {_hA} vs {_hB} — training for this regime is not bitwise deterministic'
    GATE2_MAXD, GATE2_KIND = 0.0, 'within_session_determinism'
    print(f'★ GATE2 PASS [{PANEL_REGIME}] — same-session bitwise determinism ({_hA}) (this regime has no v3 six-rule reference)')

EXP_DIR = f'{DRIVE_REPO}/checkpoints/conformal_controls_{PANEL_REGIME}'; os.makedirs(EXP_DIR, exist_ok=True)
CTRL_BASE.to_csv(f'{EXP_DIR}/controls_base_{PANEL_REGIME}.csv', index=False)
np.savez(f'{EXP_DIR}/dpm_base.npz', **{f'k{k}|{m}': v for (a, k, m), v in DPM.items() if a == 'base'})
print('saved →', f'{EXP_DIR}/controls_base_{PANEL_REGIME}.csv')


### H3. Evaluate aggressive and intermediate policy arms


### Policy-specific loss budgets

Each arm calibrates its loss budget on its own validation results at full trust.


In [ ]:
# ── H3. two-arm extension + GATE3 adaptation + firing trigger determination + enabler (15 retrains per arm) ──
BUDGETS_ARM = globals().get('BUDGETS_ARM', {})   # P0 (v1.1): per-arm per-policy budgets registry
ARM_STORE = globals().get('ARM_STORE', {})      # v1.2: retain arm policies/grids/budgets for the V-cells (additive, no numeric effect)
def run_arm_ext(beta_arm, tag):
    pols, grids, lamval, budgets_arm = {}, {}, {}, {}
    for i, sd in enumerate(POLICY_SEEDS):
        for kappa in KAPPAS:
            if (kappa, sd) not in pols:
                p_ = train_residual(pack_tr, kappa, beta_arm, ALPHA, EPOCHS, LR, seed=sd)
                pols[(kappa, sd)] = p_
                grids[(kappa, sd)] = precompute_grids(EPS_TE, pack_te, p_, kappa)
                budgets_arm[(kappa, sd)] = calibrate_budget_bp(pack_va, EPS_VA, p_, EPS_T)  # P0 fix (v1.1): per-policy c (was BASE-arm budgets[(kappa, sd)])
                lamval[(kappa, sd)] = lam_val_static(p_, kappa, budgets_arm[(kappa, sd)])
    for kappa in KAPPAS:
        _bs = [budgets_arm[(kappa, sd)] for sd in POLICY_SEEDS]
        print(f'{tag} κ={kappa} per-policy budget c (bp): ' + ', '.join(f'{x:.2f}' for x in _bs)
              + f'   (BASE-arm c was: ' + ', '.join(f'{budgets[(kappa, s)]:.2f}' for s in POLICY_SEEDS) + ')')
    rows = {}
    for kappa in KAPPAS:
        for mode in ALL_MODES:
            paths, br, lam, lpa = [], [], [], []
            for i, sd in enumerate(POLICY_SEEDS):
                out = deploy_mode(mode, pols[(kappa, sd)], kappa, sd,
                                  grids[(kappa, sd)], lamval, budgets_arm[(kappa, sd)])  # P0 fix (v1.1)
                paths.append(out['dpnl_bp']); br.append(out['breach'])
                lam.append(float(np.nanmean(out['lam']))); lpa.append(out['lam'])
            dpm = np.mean(paths, axis=0)
            m, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
            t, tl, th, _ = full_stat_ci(dpm, DATES_TE, cvar5)
            rows[(kappa, mode)] = dict(dpm=dpm, m=m, lo=lo, hi=hi, p=p,
                                       breach=float(np.nanmean(br)), lam=float(np.mean(lam)),
                                       tail=t, tl=tl, th=th, lam_path=np.mean(np.vstack(lpa), axis=0))
            print(f'{tag} κ={kappa} {mode:12s}: ΔPnL={m:+.2f} [{lo:+.2f},{hi:+.2f}] '
                  f'breach={np.nanmean(br):.3f} λ̄={np.mean(lam):.2f} tail={t:+.2f}[{tl:+.2f},{th:+.2f}]')
        f1 = rows[(kappa, 'fix1')]
        print(f'{tag} κ={kappa} prerequisite: fix1 breach={f1["breach"]:.3f} vs ε={EPS_T} → '
              + ('✓ firing scenario' if f1['breach'] > EPS_T else 'calm scenario(did not fire)'))
    BUDGETS_ARM[tag] = budgets_arm   # P0 (v1.1): recorded into manifest
    ARM_STORE[tag] = dict(pols=pols, grids=grids, lamval=lamval, budgets=budgets_arm)  # v1.2: reuse without retraining
    return rows, lamval

ARM_RESULTS, LAMVALS = {}, {}
ARMS_TO_RUN = ([(0.5, '[β=0.5]', 'arm05')] if DO_MILD else []) + [(0.0, '[β=0]', 'arm00')]
for beta_arm, tag, akey in ARMS_TO_RUN:
    print(f'\n━━ {tag} arm (β={beta_arm}) ━━')
    ARM_RESULTS[akey], LAMVALS[akey] = run_arm_ext(beta_arm, tag)
ARM0 = ARM_RESULTS['arm00']

# —— GATE3 adaptive ——
if HAS_V3_SIXARMS and 'arm05' in ARM_RESULTS and 'arm00' in ARM_RESULTS:
    import pickle
    with open(f'{V3_DIR}/arms_mix_test2022.pkl', 'rb') as fh:
        REF_ARMS = pickle.load(fh)
    GATE3_MODES = [m for m in V3_ARM5 if m.startswith('fix')]  # P0 (v1.1): budget-dependent rules legitimately move under per-policy budgets; bitwise gate keeps fixed-λ only
    GATE3_MAXD = 0.0
    for akey, rk in [('05', ARM_RESULTS['arm05']), ('00', ARM_RESULTS['arm00'])]:
        for kappa in KAPPAS:
            for mode in GATE3_MODES:
                ref = np.asarray(REF_ARMS[(akey, kappa, mode)]['dpm'], float)
                GATE3_MAXD = max(GATE3_MAXD, float(np.max(np.abs(rk[(kappa, mode)]['dpm'] - ref))))
    assert GATE3_MAXD < 1e-12, f'GATE3 fail: fixed-λ dpm maxdiff={GATE3_MAXD:.3e}'
    GATE3_KIND = 'cross_session_vs_v3 (fixed-λ only; P0 per-policy budgets)'
    print(f'★ GATE3 PASS — fixed-λ rules reproduced bitwise, maxdiff={GATE3_MAXD:.3e} (lineage locked)')
    for akey, rk in [('05', ARM_RESULTS['arm05']), ('00', ARM_RESULTS['arm00'])]:  # info only: expected divergence under per-policy budgets
        for kappa in KAPPAS:
            for mode in [m for m in V3_ARM5 if not m.startswith('fix')]:
                ref = np.asarray(REF_ARMS[(akey, kappa, mode)]['dpm'], float)
                print(f'   [info] arm{akey} κ={kappa} {mode:12s}: mean dpm shift vs v3(base-budget) = '
                      f'{float(np.mean(rk[(kappa, mode)]["dpm"] - ref)):+.3f} bp (expected ≠ 0)')
else:
    GATE3_MAXD, GATE3_KIND = float('nan'), 'n/a (this regime has no v3 two-arm reference; lineage is determined by GATE1+GATE2 locked)'
    print(f'GATE3 skip [{PANEL_REGIME}] — no v3 two-arm sealed artifacts')

# —— firing triggered + enabler (guarded wild arm crc_rwc_aci vs base arm dp_<regime>) ——
FIRE, ENABLER = {}, {}
for kappa in KAPPAS:
    FIRE[kappa] = bool(ARM0[(kappa, 'fix1')]['breach'] > EPS_T)
    g_ = ARM0[(kappa, 'crc_rwc_aci')]
    b_ = np.load(f'{UNI_DIR}/dp_{PANEL_REGIME}_k{kappa}.npy')
    dm, dl, dh, dp_ = full_stat_ci(g_['dpm'] - b_, DATES_TE, np.mean)
    ENABLER[kappa] = dict(guarded=float(g_['m']), breach=float(g_['breach']),
                          delta=float(dm), lo=float(dl), hi=float(dh), p=float(dp_))
    print(f'enabler κ={kappa}: guarded wild arm={g_["m"]:+.2f}bp breach={g_["breach"]:.3f} fire={FIRE[kappa]} | '
          f'vs base arm Δ={dm:+.2f} [{dl:+.2f},{dh:+.2f}] p_le0={dp_:.3f}')
if not any(FIRE.values()):
    print(f'\n⚠ {PANEL_REGIME}: β=0 firing not triggered (fix1 breach ≤ ε). option: '
          '(a) report truthfully"guardrail correctly lets it pass,no danger source"; '
          '(b) use the W-adversarial arm from run_cvar_wadv_train to create a firing scenario, then rerun H3.')

# —— saved to disk ——
_arm_rows = []
for akey, rows in ARM_RESULTS.items():
    for (kappa, mode), v in rows.items():
        _arm_rows.append({'arm': akey, 'kappa': kappa, 'mode': mode, 'full': v['m'], 'lo': v['lo'],
                          'hi': v['hi'], 'p_le0': v['p'], 'breach': v['breach'], 'lam': v['lam'],
                          'tail': v['tail'], 'tail_lo': v['tl'], 'tail_hi': v['th']})
CTRL_ARMS = pd.DataFrame(_arm_rows)
CTRL_ARMS.to_csv(f'{EXP_DIR}/controls_arms_{PANEL_REGIME}.csv', index=False)
for akey, rows in ARM_RESULTS.items():
    np.savez(f'{EXP_DIR}/dpm_{akey}.npz', **{f'k{k}|{m}': v['dpm'] for (k, m), v in rows.items()})
np.savez(f'{EXP_DIR}/lampath_arm00.npz', **{f'k{k}|{m}': v['lam_path'] for (k, m), v in ARM0.items()})
print('saved →', f'{EXP_DIR}/controls_arms_{PANEL_REGIME}.csv  (+npz)')


### H4. Breach-target sensitivity and trust paths


In [ ]:
# ── H4. ε frontier plot + λ_t paths (figure text in English, directly usable in the paper) ──
FIXSEQ = ['fix0', 'fix25', 'fix50', 'fix75', 'fix1']
OTHERS = ['valstatic', 'ivrule', 'emp', 'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
_MK = {'valstatic': ('s', 'tab:brown'), 'ivrule': ('P', 'tab:purple'), 'emp': ('X', 'tab:orange'),
       'a2margin': ('v', 'tab:gray'), 'aci_open': ('^', 'tab:pink'), 'crc': ('D', 'tab:cyan'),
       'crc_rwc': ('d', 'tab:blue'), 'crc_rwc_aci': ('*', 'tab:red')}

def _get(df, arm, kappa, mode):
    r = df[(df['arm'] == arm) & (df['kappa'] == kappa) & (df['mode'] == mode)].iloc[0]
    return float(r['breach']), float(r['full'])

def frontier_fig(df, arm, fname, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    for ax, kappa in zip(axes, KAPPAS):
        xs, ys = zip(*[_get(df, arm, kappa, m) for m in FIXSEQ])
        ax.plot(xs, ys, '-o', color='k', lw=1, ms=4, alpha=0.6, label='fixed λ (0→1)', zorder=2)
        for la, x, y in zip([0, .25, .5, .75, 1], xs, ys):
            ax.annotate(f'{la:g}', (x, y), fontsize=7, xytext=(3, 3), textcoords='offset points')
        for m in OTHERS:
            x, y = _get(df, arm, kappa, m); mk, c = _MK[m]
            ax.scatter([x], [y], marker=mk, color=c, s=80 if m == 'crc_rwc_aci' else 42, label=m, zorder=3)
        ax.axvline(EPS_T, ls='--', lw=1, color='r', alpha=0.6)
        ax.set_title(f'kappa={kappa}'); ax.set_xlabel('realized breach rate (informed episodes)')
        ax.grid(alpha=0.25)
    axes[0].set_ylabel('mean ΔPnL vs BS (bp/episode)')
    axes[-1].legend(fontsize=7, loc='best', ncol=2)
    fig.suptitle(title, y=1.03); fig.tight_layout()
    for ext in ['png', 'pdf']:
        fig.savefig(f'{EXP_DIR}/{fname}_{PANEL_REGIME}.{ext}', dpi=300, bbox_inches='tight')
    plt.show(); print('saved →', f'{EXP_DIR}/{fname}_{PANEL_REGIME}.png/.pdf')

frontier_fig(CTRL_BASE, 'base', 'fig_frontier_base', f'Compliant arm (beta*): 13-rule epsilon-frontier — {PANEL_REGIME}')
if 'arm05' in ARM_RESULTS:
    frontier_fig(CTRL_ARMS, 'arm05', 'fig_frontier_arm05', f'Mild arm (beta=0.5): epsilon-frontier — {PANEL_REGIME}')
frontier_fig(CTRL_ARMS, 'arm00', 'fig_frontier_arm00', f'Wild arm (beta=0, fire test) — {PANEL_REGIME}')

ordr = np.argsort(DATES_TE)
fig, ax = plt.subplots(figsize=(12, 3.6))
for m, lw in [('crc_rwc_aci', 2.0), ('emp', 1.2), ('ivrule', 1.2), ('valstatic', 1.2)]:
    ax.plot(DATES_TE[ordr], ARM0[(2, m)]['lam_path'][ordr], lw=lw, alpha=0.85, label=m, color=_MK[m][1])
ax.axhline(1.0, ls=':', lw=0.8, color='k', alpha=0.5)
ax.set_ylim(-0.05, 1.1); ax.set_ylabel('lambda_t (mean over 5 seeds)'); ax.legend(fontsize=8, ncol=4)
ax.set_title(f'Wild arm, kappa=2: deployed lambda_t path — {PANEL_REGIME}')
fig.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f'{EXP_DIR}/fig_lambda_path_arm00_{PANEL_REGIME}.{ext}', dpi=300, bbox_inches='tight')
plt.show(); print('saved →', f'{EXP_DIR}/fig_lambda_path_arm00_{PANEL_REGIME}.png/.pdf')


### H5. Save regime results


In [ ]:
# ── H5. this regime's control group manifest ──
import hashlib as _hh, json as _jj, glob as _gg, time as _tt
_env = {'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
        'threads': torch.get_num_threads(), 'device': str(DEVICE)}
_files = {os.path.basename(f): _file_md5(f) for f in sorted(_gg.glob(f'{EXP_DIR}/*'))
          if not f.endswith('.json')}
_man = {'created': _tt.strftime('%Y-%m-%d %H:%M:%S'), 'regime': PANEL_REGIME, 'env': _env,
        'gates': {'gate1_dp_bitwise': bool(GATE1),
                  'gate2_kind': GATE2_KIND, 'gate2_maxdiff': float(GATE2_MAXD),
                  'gate3_kind': GATE3_KIND, 'gate3_maxdiff': float(GATE3_MAXD)},
        'eps_target': EPS_T, 'lam_grid': LAM_GRID, 'do_mild': bool(DO_MILD),
        'budgets_arm_bp': {f'{a}|{k}|{s}': round(float(v), 4) for a, d in globals().get('BUDGETS_ARM', {}).items() for (k, s), v in d.items()},  # P0 (v1.1)
        'fire': {str(k): bool(FIRE[k]) for k in KAPPAS},
        'enabler': {str(k): ENABLER[k] for k in KAPPAS},
        'lam_val_base': {f'{k}|{s}': LAMVAL_BASE[(k, s)] for k in KAPPAS for s in POLICY_SEEDS},
        'upstream_dp': {f'k{k}': _file_md5(f'{UNI_DIR}/dp_{PANEL_REGIME}_k{k}.npy') for k in KAPPAS},
        'files': _files}
with open(f'{EXP_DIR}/controls_manifest_{PANEL_REGIME}.json', 'w') as fh:
    _jj.dump(_man, fh, ensure_ascii=False, indent=1)
print('=' * 76)
print(f'★ [{PANEL_REGIME}] control group complete @ {_man["created"]}')
print(f'  GATE1={GATE1} | GATE2={GATE2_KIND}({GATE2_MAXD:.1e}) | GATE3={GATE3_KIND}')
print(f'  fire: {{k:FIRE[k]}}'.replace('{k:FIRE[k]}', str({k: FIRE[k] for k in KAPPAS})))
print(f'  → {EXP_DIR}/controls_manifest_{PANEL_REGIME}.json')
print('  switch to the next regime: change the H0 selector → restart the kernel → rerun the whole notebook. After all four are done, run H6.')
print('=' * 76)


## Common-budget evaluation

Compare both policy arms under a 5 bp loss budget and a target breach probability of 0.1.


In [ ]:
# ── V1 (v1.2). λ-grid ledger + budget-parameterised replay machinery ──
# Additive layer: nothing here retrains or modifies H1–H3 results. The ledger stores, per policy,
# the full counterfactual grids ΔPnL_i(λ) (test + validation) and Δreward_i(λ) (test), so that ANY
# budget c can be replayed deterministically without touching the networks (§4.3.3 / §4.3.8).
C_COMMON  = 5.0                      # unified budget (bp) = the BASE arm's certified standard
C_SWEEP   = [5.0, 7.5, 10.0, 15.0]   # optional budget sweep (V4)
LOAD_FROM_LEDGER = False             # True → skip H1–H3, rebuild everything from ledger npz on disk

SS_VA = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_VA.iterrows()])

def _v12_md5(p):
    import hashlib as _l
    h = _l.md5()
    with open(p, 'rb') as fh:
        for ch in iter(lambda: fh.read(1 << 20), b''): h.update(ch)
    return h.hexdigest()[:12]

def build_ledger_from_session(pol, kappa):
    # identical inputs to H2/H3: precompute_grids on test, plus the validation dp grid
    g = dict(precompute_grids(EPS_TE, pack_te, pol, kappa))
    g['vadp'] = {la: episode_dpnl_bp_lambda(pack_va, pol, np.full(len(EPS_VA), la), SS_VA)
                 for la in LAM_GRID}
    return g

def save_ledger(store, arm_key):
    # store: {(kappa, sd): grids-with-vadp}; timing/context arrays are policy-independent → stored once
    any_g = next(iter(store.values()))
    payload = {'ds': any_g['ds'].astype('datetime64[ns]').astype('int64'),
               'de': any_g['de'].astype('datetime64[ns]').astype('int64'),
               'dte': DATES_TE.astype('datetime64[ns]').astype('int64'),
               'iv': any_g['iv'], 'ss': any_g['ss'], 'order': any_g['order']}
    for (k, sd), g in store.items():
        for la in LAM_GRID:
            payload[f'k{k}|s{sd}|dp|{la}']   = g['dp_lam'][la]
            payload[f'k{k}|s{sd}|d|{la}']    = g['d_lam'][la]
            payload[f'k{k}|s{sd}|vadp|{la}'] = g['vadp'][la]
    out = f'{EXP_DIR}/ledger_{arm_key}_{PANEL_REGIME}.npz'
    np.savez_compressed(out, **payload)
    print(f'  ledger saved → {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
    return out

def load_ledger(arm_key):
    z = np.load(f'{EXP_DIR}/ledger_{arm_key}_{PANEL_REGIME}.npz')
    shared = dict(ds=z['ds'].astype('datetime64[ns]'), de=z['de'].astype('datetime64[ns]'),
                  iv=z['iv'], ss=z['ss'], order=z['order'])
    n = len(shared['ds']); store = {}
    for key in z.files:
        if key.count('|') != 3: continue
        kk, sk, kind, la = key.split('|')
        g = store.setdefault((int(kk[1:]), int(sk[1:])),
                             dict(n=n, d_lam={}, dp_lam={}, vadp={}, **shared))
        {'dp': g['dp_lam'], 'd': g['d_lam'], 'vadp': g['vadp']}[kind][float(la)] = z[key]
    return store

def deploy_from_ledger(L, mode='crc_rwc', eps_target=0.10, budget_bp=25.0,
                       gamma_eps=0.02, half_life=30.0, tau_iv=0.04, warmup=15,
                       M_emp=0.3, lam_warmup=1.0, lam_fix=None):
    # faithful torch-free copy of G2.deploy_lambda_control: identical decision loop; final evaluation
    # by grid lookup (elementwise ops + per-episode segment sums make the mixed-λ torch evaluation
    # equal the fixed-λ lookup bitwise). Certified against the pipeline by GATE4 (V2) — do not edit
    # this function without re-running GATE4 on every regime.
    n, ds, de, iv, order = L['n'], L['ds'], L['de'], L['iv'], L['order']
    d_lam, dp_lam = L['d_lam'], L['dp_lam']
    used = np.full(n, np.nan); informed = np.zeros(n, bool)
    lam_aci = float(lam_warmup); eps_eff = float(eps_target)
    aci_pending = []
    for t, i in enumerate(order):
        past = order[:t]
        done = past[de[past] < ds[i]]
        informed[i] = len(done) >= warmup
        if mode == 'crc_rwc_aci' and aci_pending:
            still = []
            for j in aci_pending:
                if de[j] < ds[i]:
                    err = float(dp_lam[used[j]][j] < -budget_bp)
                    eps_eff = float(np.clip(eps_eff + gamma_eps * (eps_target - err),
                                            0.25 * eps_target, min(0.45, 4 * eps_target)))
                else:
                    still.append(j)
            aci_pending = still
        if   mode == 'fix1': lam_t = 1.0
        elif mode == 'fix0': lam_t = 0.0
        elif mode == 'fixv': lam_t = float(lam_fix)
        elif not informed[i]:
            lam_t = 0.0 if mode == 'a2margin' else lam_warmup
        elif mode == 'a2margin':
            thr = cvar_5(d_lam[0.0][done]) - M_emp
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if cvar_5(d_lam[la][done]) >= thr: lam_t = la; break
        elif mode == 'emp':
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if float(np.mean(dp_lam[la][done] < -budget_bp)) <= eps_target:
                    lam_t = la; break
        elif mode == 'ivrule':
            p_iv = float(np.mean(iv[done] <= iv[i]))
            lam_t = 1.0 if p_iv < 0.5 else (0.5 if p_iv < 0.8 else 0.25)
        elif mode == 'aci_open':
            var_hat = np.quantile(dp_lam[1.0][done], eps_target)
            err = float(dp_lam[1.0][i] < var_hat)
            lam_aci = float(np.clip(lam_aci + 0.10 * (eps_target - err), 0.0, 1.0))
            lam_t = lam_aci
        else:
            if mode in ('crc_rwc', 'crc_rwc_aci'):
                dt_ = (ds[i] - de[done]) / np.timedelta64(1, 'D')
                w = (np.exp(-np.log(2) * np.maximum(dt_, 0.0) / half_life)
                     * np.exp(-np.abs(iv[done] - iv[i]) / tau_iv))
            else:
                w = np.ones(len(done))
            e_use = eps_eff if mode == 'crc_rwc_aci' else eps_target
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if conformal_lower_q(dp_lam[la][done], w, e_use) >= -budget_bp:
                    lam_t = la; break
        used[i] = lam_t
        if mode == 'crc_rwc_aci': aci_pending.append(i)
    d_used  = np.array([d_lam[used[i]][i]  for i in range(n)], dtype=d_lam[1.0].dtype)
    dp_used = np.array([dp_lam[used[i]][i] for i in range(n)], dtype=dp_lam[1.0].dtype)
    m = informed
    breach = float(np.mean(dp_used[m] < -budget_bp)) if m.any() else np.nan
    return {'lam': used, 'd': d_used, 'dpnl_bp': dp_used, 'breach': breach, 'informed': m,
            'budget_bp': float(budget_bp), 'eps_eff_final': float(eps_eff)}

def lam_val_static_from_ledger(g, budget_bp, eps_target=EPS_T):
    # mirrors G2.lam_val_static, on the stored validation dp grid
    for la in sorted(LAM_GRID, reverse=True):
        if float(np.mean(g['vadp'][la] < -budget_bp)) <= eps_target:
            return float(la)
    return 0.0

def budget_from_ledger(g, eps_target=EPS_T, floor_bp=5.0):
    # mirrors G2.calibrate_budget_bp, on the stored validation dp grid at λ=1
    return float(max(floor_bp, -np.quantile(g['vadp'][1.0], eps_target)))

def replay_mode(g, mode, budget_bp, lamval_entry, pol=None, kappa=None):
    # mirrors G2.deploy_mode dispatch. Exception: aci_open updates λ continuously OFF the five-point
    # grid, so it cannot be replayed from the ledger alone — it is routed through the ORIGINAL
    # pipeline function with the session policy (exact v1.1 code path, zero transcription risk).
    # Ledger-only sessions skip it: it is an ablation control, not part of the §7.6 deliverable.
    if mode == 'aci_open':
        assert pol is not None, 'aci_open needs a session policy (off-grid λ) — LOAD_FROM_LEDGER sessions skip it'
        return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode='aci_open',
                                     eps_target=EPS_T, budget_bp=budget_bp, grids=g)
    if mode in ('fix25', 'fix50', 'fix75'):
        return deploy_from_ledger(g, mode='fixv', lam_fix=float(mode[3:]) / 100.0,
                                  eps_target=EPS_T, budget_bp=budget_bp)
    if mode == 'valstatic':
        return deploy_from_ledger(g, mode='fixv', lam_fix=lamval_entry,
                                  eps_target=EPS_T, budget_bp=budget_bp)
    return deploy_from_ledger(g, mode=mode, eps_target=EPS_T, budget_bp=budget_bp)

# ── assemble ledgers ──
LEDGERS = {}
if not LOAD_FROM_LEDGER:
    assert 'BASE' in globals() and 'ARM_STORE' in globals() and len(ARM_STORE), \
        'run H1–H3 first (or set LOAD_FROM_LEDGER=True with ledger npz present)'
    print('building ledgers from session policies (deterministic re-derivation, no retraining)...')
    LEDGERS['base'] = {(k, sd): build_ledger_from_session(BASE[k][i]['pol'], k)
                       for k in KAPPAS for i, sd in enumerate(POLICY_SEEDS)}
    _TAG2KEY = {tag: akey for _, tag, akey in ARMS_TO_RUN}
    for tag, st in ARM_STORE.items():
        akey = _TAG2KEY[tag]; LEDGERS[akey] = {}
        for (k, sd), g0 in st['grids'].items():
            g = dict(g0)
            g['vadp'] = {la: episode_dpnl_bp_lambda(pack_va, st['pols'][(k, sd)],
                                                    np.full(len(EPS_VA), la), SS_VA)
                         for la in LAM_GRID}
            LEDGERS[akey][(k, sd)] = g
    for akey, store in LEDGERS.items():
        save_ledger(store, akey)
else:
    for akey in ['base'] + (['arm05'] if DO_MILD else []) + ['arm00']:
        _f = f'{EXP_DIR}/ledger_{akey}_{PANEL_REGIME}.npz'
        if os.path.exists(_f): LEDGERS[akey] = load_ledger(akey)
        else: print(f'  (no ledger for {akey} — skipped)')
    print('ledgers loaded:', {k: len(v) for k, v in LEDGERS.items()})
POLS = {}
if not LOAD_FROM_LEDGER:
    for k in KAPPAS:
        for i, sd in enumerate(POLICY_SEEDS):
            POLS[('base', k, sd)] = BASE[k][i]['pol']
    for tag, st in ARM_STORE.items():
        for (k, sd), p_ in st['pols'].items():
            POLS[(_TAG2KEY[tag], k, sd)] = p_
print('V1 ready — arms:', sorted(LEDGERS), f'| C_COMMON = {C_COMMON:g} bp | C_SWEEP = {C_SWEEP}'
      + (' | ledger-only mode: aci_open will be skipped' if LOAD_FROM_LEDGER else ''))

In [ ]:
# ── V2 (v1.2). GATE4: ledger replay ≡ deployment pipeline, bitwise ──
# Replays every (arm × κ × seed × 13 rules) from the ledger at the SAME budgets used by H2/H3 and
# asserts across-seed dpm / λ-paths (and recomputed budgets/λ_val) match the session results exactly.
# After this gate, any difference seen at another budget can only come from the budget itself.
_g4_maxd, _g4_n = 0.0, 0
def _g4_check(a, b, tagtxt):
    global _g4_maxd, _g4_n
    a, b = np.asarray(a, float), np.asarray(b, float)
    same = np.array_equal(a, b)
    if not same: _g4_maxd = max(_g4_maxd, float(np.nanmax(np.abs(a - b))))
    _g4_n += 1
    assert same, f'GATE4 fail @ {tagtxt}: maxdiff={np.nanmax(np.abs(a - b)):.3e}'

if not LOAD_FROM_LEDGER:
    _BUD  = {'base': dict(budgets)}
    _LAMV = {'base': dict(LAMVAL_BASE)}
    for tag, st in ARM_STORE.items():
        _BUD[_TAG2KEY[tag]] = st['budgets']; _LAMV[_TAG2KEY[tag]] = st['lamval']
    # (i) budgets and λ_val recomputed from the ledger must equal the pipeline's
    for akey, store in LEDGERS.items():
        for (k, sd), g in store.items():
            _g4_check([budget_from_ledger(g)], [_BUD[akey][(k, sd)]], f'{akey} budget κ={k} s={sd}')
            _g4_check([lam_val_static_from_ledger(g, _BUD[akey][(k, sd)])],
                      [_LAMV[akey][(k, sd)]], f'{akey} λ_val κ={k} s={sd}')
    # (ii) full 13-rule replay vs session results
    _REF = {'base': {(k, m): dict(dpm=DPM[('base', k, m)], lam_path=LPATH[('base', k, m)])
                     for k in KAPPAS for m in ALL_MODES}}
    for tag, st in ARM_STORE.items():
        _REF[_TAG2KEY[tag]] = ARM_RESULTS[_TAG2KEY[tag]]
    for akey, store in LEDGERS.items():
        for k in KAPPAS:
            for mode in ALL_MODES:
                paths, lpa = [], []
                for sd in POLICY_SEEDS:
                    out = replay_mode(store[(k, sd)], mode, _BUD[akey][(k, sd)], _LAMV[akey][(k, sd)],
                                      pol=POLS.get((akey, k, sd)), kappa=k)
                    paths.append(out['dpnl_bp']); lpa.append(out['lam'])
                _g4_check(np.mean(paths, axis=0), _REF[akey][(k, mode)]['dpm'],
                          f'{akey} κ={k} {mode} dpm')
                _g4_check(np.mean(np.vstack(lpa), axis=0), _REF[akey][(k, mode)]['lam_path'],
                          f'{akey} κ={k} {mode} λpath')
            print(f'  GATE4 ✓ {akey} κ={k}: 13 rules × {len(POLICY_SEEDS)} seeds bitwise')
    GATE4_KIND = 'in_session_ledger_replay_vs_pipeline'
else:
    # fresh session: compare across-seed means against the sealed artifacts on disk
    _BUD, _LAMV = {}, {}
    for akey, store in LEDGERS.items():
        _BUD[akey]  = {ks: budget_from_ledger(g) for ks, g in store.items()}
        _LAMV[akey] = {ks: lam_val_static_from_ledger(g, _BUD[akey][ks]) for ks, g in store.items()}
    for akey, store in LEDGERS.items():
        ref = np.load(f'{EXP_DIR}/dpm_{akey}.npz')
        for k in KAPPAS:
            for mode in ALL_MODES:
                if mode == 'aci_open': continue     # off-grid λ — needs session policies; skipped in ledger-only mode
                paths = [replay_mode(store[(k, sd)], mode, _BUD[akey][(k, sd)],
                                     _LAMV[akey][(k, sd)])['dpnl_bp'] for sd in POLICY_SEEDS]
                _g4_check(np.mean(paths, axis=0), ref[f'k{k}|{mode}'], f'{akey} κ={k} {mode} dpm(disk)')
            print(f'  GATE4 ✓ {akey} κ={k}: 12 rules vs disk artifacts bitwise (aci_open needs session policies — skipped)')
    GATE4_KIND = 'ledger_replay_vs_disk_artifacts'
GATE4, GATE4_MAXD = True, float(_g4_maxd)
print(f'★ GATE4 PASS [{PANEL_REGIME}] — {_g4_n} comparisons bitwise identical ({GATE4_KIND})')

In [ ]:
# ── V3 (v1.2). Unified-budget evaluation: both arms, same rule, c = C_COMMON, ε = EPS_T ──
# Ce's design for §7.6: hold BOTH arms to the conservative arm's certified standard (c = 5 bp),
# deploy both under the same rule family, keep ε and every ACI parameter untouched — the only
# remaining difference between the arms is the training objective (β* vs β = 0).
assert 'GATE4' in globals() and GATE4, 'run V2 (GATE4) first'
LAMVAL_C5, ROWS_C5, DPM_C5, LPATH_C5 = {}, [], {}, {}
for akey, store in LEDGERS.items():
    for (k, sd), g in store.items():
        LAMVAL_C5[(akey, k, sd)] = lam_val_static_from_ledger(g, C_COMMON)
for akey, store in LEDGERS.items():
    for k in KAPPAS:
        for mode in ALL_MODES:
            if LOAD_FROM_LEDGER and mode == 'aci_open':
                print(f'[c={C_COMMON:g}] {akey:5s} κ={k} aci_open     : skipped (off-grid λ — needs session policies)')
                continue
            paths, br, lam, lpa = [], [], [], []
            for sd in POLICY_SEEDS:
                out = replay_mode(store[(k, sd)], mode, C_COMMON, LAMVAL_C5[(akey, k, sd)],
                                  pol=POLS.get((akey, k, sd)), kappa=k)
                paths.append(out['dpnl_bp']); br.append(out['breach'])
                lam.append(float(np.nanmean(out['lam']))); lpa.append(out['lam'])
            dpm = np.mean(paths, axis=0)
            DPM_C5[(akey, k, mode)] = dpm
            LPATH_C5[(akey, k, mode)] = np.mean(np.vstack(lpa), axis=0)
            m_, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
            t, tl, th, _ = full_stat_ci(dpm, DATES_TE, cvar5)
            ROWS_C5.append({'arm': akey, 'kappa': k, 'mode': mode, 'budget_bp': C_COMMON,
                            'full': m_, 'lo': lo, 'hi': hi, 'p_le0': p,
                            'breach': float(np.nanmean(br)), 'lam': float(np.mean(lam)),
                            'tail': t, 'tail_lo': tl, 'tail_hi': th})
            print(f'[c={C_COMMON:g}] {akey:5s} κ={k} {mode:12s}: ΔPnL={m_:+.2f} [{lo:+.2f},{hi:+.2f}] '
                  f'breach={np.nanmean(br):.3f} λ̄={np.mean(lam):.2f} tail={t:+.1f}')
CTRL_C5 = pd.DataFrame(ROWS_C5)

# consistency: the base arm is floor-bound at 5 bp in every cell → its c5 numbers must equal H2's
if (not LOAD_FROM_LEDGER) and all(abs(budgets[ks] - C_COMMON) < 1e-12 for ks in budgets):
    for k in KAPPAS:
        for mode in ALL_MODES:
            assert np.array_equal(DPM_C5[('base', k, mode)], DPM[('base', k, mode)]), \
                f'base c5 ≠ H2 @ κ={k} {mode} (impossible under identical budget — investigate)'
    print('✓ base arm @ c5 ≡ H2 per-policy results (floor-bound at 5 bp): the conservative arm is unchanged')

# ── the §7.6 comparison per κ: symmetric guarded-vs-guarded (primary) + vs base fix1 (secondary) ──
ENABLER_C5, FIRE_C5 = {}, {}
for k in KAPPAS:
    gA = DPM_C5[('arm00', k, 'crc_rwc_aci')]; gB = DPM_C5[('base', k, 'crc_rwc_aci')]
    dm, dl, dh, dp_ = full_stat_ci(gA - gB, DATES_TE, np.mean)
    b_fix1 = np.load(f'{UNI_DIR}/dp_{PANEL_REGIME}_k{k}.npy')
    dm2, dl2, dh2, dp2 = full_stat_ci(gA - b_fix1, DATES_TE, np.mean)
    rowA = CTRL_C5[(CTRL_C5['arm'] == 'arm00') & (CTRL_C5['kappa'] == k)].set_index('mode')
    rowB = CTRL_C5[(CTRL_C5['arm'] == 'base') & (CTRL_C5['kappa'] == k)].set_index('mode')
    FIRE_C5[k] = bool(rowA.loc['fix1', 'breach'] > EPS_T)
    ENABLER_C5[k] = dict(
        aggr_guarded=float(rowA.loc['crc_rwc_aci', 'full']),
        aggr_breach=float(rowA.loc['crc_rwc_aci', 'breach']),
        aggr_lam=float(rowA.loc['crc_rwc_aci', 'lam']),
        aggr_tail=float(rowA.loc['crc_rwc_aci', 'tail']),
        cons_guarded=float(rowB.loc['crc_rwc_aci', 'full']),
        cons_breach=float(rowB.loc['crc_rwc_aci', 'breach']),
        cons_tail=float(rowB.loc['crc_rwc_aci', 'tail']),
        unguarded_aggr_breach=float(rowA.loc['fix1', 'breach']),
        delta_sym=float(dm), lo_sym=float(dl), hi_sym=float(dh), p_sym=float(dp_),
        delta_vs_basefix1=float(dm2), lo_vs_basefix1=float(dl2),
        hi_vs_basefix1=float(dh2), p_vs_basefix1=float(dp2))
    e = ENABLER_C5[k]
    print(f'\n[c={C_COMMON:g}] κ={k}  aggressive(guarded) {e["aggr_guarded"]:+.2f}bp @breach {e["aggr_breach"]:.3f} '
          f'λ̄={e["aggr_lam"]:.2f} | conservative(guarded) {e["cons_guarded"]:+.2f}bp @ {e["cons_breach"]:.3f}')
    print(f'          Δ(guarded−guarded) = {e["delta_sym"]:+.2f} [{e["lo_sym"]:+.2f},{e["hi_sym"]:+.2f}] '
          f'p_le0={e["p_sym"]:.3f} | Δ(vs base fix1) = {e["delta_vs_basefix1"]:+.2f} p={e["p_vs_basefix1"]:.3f}')
    print(f'          unguarded aggressive breach@c={C_COMMON:g}: {e["unguarded_aggr_breach"]:.3f} '
          f'| tails: aggr {e["aggr_tail"]:+.1f} vs cons {e["cons_tail"]:+.1f} bp')

# ── save (v1.1 artifacts untouched — every output carries the _c5 suffix) ──
_c5tag = f'c{C_COMMON:g}'.replace('.', 'p')
CTRL_C5[CTRL_C5['arm'] == 'base'].to_csv(f'{EXP_DIR}/controls_base_{PANEL_REGIME}_{_c5tag}.csv', index=False)
CTRL_C5[CTRL_C5['arm'] != 'base'].to_csv(f'{EXP_DIR}/controls_arms_{PANEL_REGIME}_{_c5tag}.csv', index=False)
for akey in LEDGERS:
    np.savez(f'{EXP_DIR}/dpm_{akey}_{_c5tag}.npz',
             **{f'k{k}|{m}': DPM_C5[(akey, k, m)] for k in KAPPAS for m in ALL_MODES
                if (akey, k, m) in DPM_C5})
np.savez(f'{EXP_DIR}/lampath_arm00_{_c5tag}.npz',
         **{f'k{k}|{m}': LPATH_C5[('arm00', k, m)] for k in KAPPAS for m in ALL_MODES
            if ('arm00', k, m) in LPATH_C5})
import json as _jj, time as _tt
_man5 = {'created': _tt.strftime('%Y-%m-%d %H:%M:%S'), 'regime': PANEL_REGIME,
         'budget_mode': 'common', 'c_common_bp': C_COMMON, 'eps_target': EPS_T,
         'lam_grid': LAM_GRID, 'arms': sorted(LEDGERS),
         'aci_params': {'gamma': 0.02, 'eps_clip': [0.25 * EPS_T, min(0.45, 4 * EPS_T)],
                        'half_life_days': 30.0, 'tau_iv': 0.04, 'warmup': 15},
         'gates': {'gate4_kind': GATE4_KIND, 'gate4_bitwise': bool(GATE4),
                   'gate4_comparisons': int(_g4_n)},
         'budgets_per_policy_reference': {f'{a}|{k}|{s}': round(float(_BUD[a][(k, s)]), 4)
                                          for a in _BUD for (k, s) in _BUD[a]},
         'lamval_c5': {f'{a}|{k}|{s}': v for (a, k, s), v in LAMVAL_C5.items()},
         'fire_c5': {str(k): FIRE_C5[k] for k in KAPPAS},
         'enabler_c5': {str(k): ENABLER_C5[k] for k in KAPPAS},
         'files': {os.path.basename(f): _v12_md5(f)
                   for f in sorted(glob.glob(f'{EXP_DIR}/*_{_c5tag}*')) if not f.endswith('.json')}}
with open(f'{EXP_DIR}/controls_manifest_{PANEL_REGIME}_{_c5tag}.json', 'w') as fh:
    _jj.dump(_man5, fh, ensure_ascii=False, indent=1)
print(f'\nsaved → controls_(base|arms)_{PANEL_REGIME}_{_c5tag}.csv + dpm/lampath npz + manifest')

In [ ]:
# ── V4 (v1.2, optional). Budget sweep from the ledger — where does the ordering move? ──
# Pure post-processing (no training, no torch). The wild arm is redeployed at each c; the reference
# is the conservative arm at ITS certified standard (guarded @ 5 bp), which is the fixed yardstick.
# fix1's deployment path is budget-independent — only its breach metric moves with c.
_srows = []
for c_ in C_SWEEP:
    for k in KAPPAS:
        paths, br, lam, f1_br = [], [], [], []
        for sd in POLICY_SEEDS:
            lam_c = lam_val_static_from_ledger(LEDGERS['arm00'][(k, sd)], c_)
            out = replay_mode(LEDGERS['arm00'][(k, sd)], 'crc_rwc_aci', c_, lam_c)
            paths.append(out['dpnl_bp']); br.append(out['breach'])
            lam.append(float(np.nanmean(out['lam'])))
            f1_br.append(replay_mode(LEDGERS['arm00'][(k, sd)], 'fix1', c_, lam_c)['breach'])
        dpm = np.mean(paths, axis=0)
        gB = DPM_C5[('base', k, 'crc_rwc_aci')]
        m_, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
        dm, dl, dh, dp_ = full_stat_ci(dpm - gB, DATES_TE, np.mean)
        _srows.append(dict(c_bp=c_, kappa=k, mean=m_, lo=lo, hi=hi,
                           breach=float(np.nanmean(br)), lam=float(np.mean(lam)), tail=cvar5(dpm),
                           fix1_breach=float(np.nanmean(f1_br)),
                           delta_vs_cons_guarded5=dm, d_lo=dl, d_hi=dh, d_p=dp_))
        print(f'c={c_:>5.1f}bp κ={k}: crc_rwc_aci {m_:+.2f} [{lo:+.2f},{hi:+.2f}] '
              f'breach={np.nanmean(br):.3f} λ̄={np.mean(lam):.2f} | '
              f'Δ vs cons(guarded@5) {dm:+.2f} p={dp_:.3f} | fix1 breach {np.nanmean(f1_br):.3f}')
SWEEP = pd.DataFrame(_srows)
SWEEP.to_csv(f'{EXP_DIR}/budget_sweep_{PANEL_REGIME}_arm00.csv', index=False)
print('saved →', f'{EXP_DIR}/budget_sweep_{PANEL_REGIME}_arm00.csv')

### H6. Combine the regime results


In [ ]:
# ── H6. panel aggregation: 4-regime × 3κ firing panel + manifest (can be run first on any subset) ──
import json as _jj, time as _tt
PANEL_REGIMES = ['crash_test2018', 'crash_test2020', 'mix_test2022', 'mix_test2024']
SIMPLE_FAIL  = ['fix50', 'fix75', 'fix1', 'valstatic', 'ivrule', 'aci_open']  # "fire-and-forget" control
prows = []
for RG in PANEL_REGIMES:
    d = f'{DRIVE_REPO}/checkpoints/conformal_controls_{RG}'
    fb, fa, fm = (f'{d}/controls_base_{RG}.csv', f'{d}/controls_arms_{RG}.csv',
                  f'{d}/controls_manifest_{RG}.json')
    if not (os.path.exists(fb) and os.path.exists(fa) and os.path.exists(fm)):
        print(f'  {RG}: missing control-group artifacts, skip (run H0-H5 for that regime first)'); continue
    arms = pd.read_csv(fa); man = _jj.load(open(fm))
    for kappa in [1, 2, 3]:
        a0 = arms[(arms['arm'] == 'arm00') & (arms['kappa'] == kappa)].set_index('mode')
        if 'crc_rwc_aci' not in a0.index or 'fix1' not in a0.index:
            continue
        win, fix1 = a0.loc['crc_rwc_aci'], a0.loc['fix1']
        ret = (win['full'] / fix1['full'] * 100.0) if abs(fix1['full']) > 1e-9 else float('nan')
        worst = float(a0.loc[[m for m in SIMPLE_FAIL if m in a0.index], 'breach'].max())
        en = man.get('enabler', {}).get(str(kappa), {})
        prows.append(dict(
            regime=RG, kappa=kappa, fire=bool(man.get('fire', {}).get(str(kappa), False)),
            crcaci_mean=round(float(win['full']), 2), crcaci_breach=round(float(win['breach']), 3),
            crcaci_tail=round(float(win['tail']), 1), retention_pct=round(ret, 0),
            fix1_mean=round(float(fix1['full']), 2), fix1_breach=round(float(fix1['breach']), 3),
            worst_simple_breach=round(worst, 3),
            held=bool(win['breach'] <= 0.10), simple_failed=bool(worst > 0.10),
            enabler_delta=round(float(en.get('delta', float('nan'))), 2),
            enabler_p=round(float(en.get('p', float('nan'))), 3)))
PANEL = pd.DataFrame(prows)
if len(PANEL):
    cols = ['regime', 'kappa', 'fire', 'crcaci_mean', 'crcaci_breach', 'crcaci_tail',
            'retention_pct', 'worst_simple_breach', 'held', 'simple_failed',
            'enabler_delta', 'enabler_p']
    print(PANEL[cols].to_string(index=False))
    n_held = int(PANEL['held'].sum()); n_total = len(PANEL)
    n_sep  = int((PANEL['held'] & PANEL['simple_failed']).sum())
    print(f'\naggregate: crc_rwc_aci breach≤ε cell = {n_held}/{n_total} | '
          f'"crc_rwc_acicomplies while the simple control breaches" cell = {n_sep}/{n_total}')
    out_csv = f'{DRIVE_REPO}/checkpoints/controls_panel_summary.csv'
    PANEL.to_csv(out_csv, index=False)
    man = {'created': _tt.strftime('%Y-%m-%d %H:%M:%S'),
           'regimes_present': sorted(PANEL['regime'].unique().tolist()),
           'eps_target': 0.10, 'n_cells': n_total, 'n_held': n_held, 'n_separated': n_sep,
           'note': 'main-table numbers still come from consolidated_v2/manifest; control-panel numbers come only from the per-regime controls_manifest_*.'}
    with open(f'{DRIVE_REPO}/checkpoints/controls_panel_manifest.json', 'w') as fh:
        _jj.dump(man, fh, ensure_ascii=False, indent=1)
    print('saved →', out_csv, '+ controls_panel_manifest.json')
else:
    print('nothing to aggregate yet — run H0-H5 for at least one regime first.')


## Combine the common-budget results

Run after the common-budget evaluation has completed for all four regimes.


In [ ]:
# ── V5 (v1.2). c5 panel aggregation: 4-regime × 3κ unified-budget comparison ──
import json as _jj, time as _tt
PANEL_REGIMES = ['crash_test2018', 'crash_test2020', 'mix_test2022', 'mix_test2024']
_c5tag5 = (f'c{C_COMMON:g}'.replace('.', 'p') if 'C_COMMON' in globals() else 'c5')
prows5 = []
for RG in PANEL_REGIMES:
    fm = f'{DRIVE_REPO}/checkpoints/conformal_controls_{RG}/controls_manifest_{RG}_{_c5tag5}.json'
    if not os.path.exists(fm):
        print(f'  {RG}: no {_c5tag5} manifest yet — run H0→V3 for that regime first'); continue
    man = _jj.load(open(fm))
    for k in [1, 2, 3]:
        e = man.get('enabler_c5', {}).get(str(k))
        if not e: continue
        prows5.append(dict(regime=RG, kappa=k,
                           cons_guarded=round(e['cons_guarded'], 2),
                           aggr_guarded=round(e['aggr_guarded'], 2),
                           delta_sym=round(e['delta_sym'], 2),
                           lo=round(e['lo_sym'], 2), hi=round(e['hi_sym'], 2),
                           p_sym=round(e['p_sym'], 4),
                           cons_breach=round(e['cons_breach'], 3),
                           aggr_breach=round(e['aggr_breach'], 3),
                           aggr_lam=round(e['aggr_lam'], 2),
                           cons_tail=round(e['cons_tail'], 1), aggr_tail=round(e['aggr_tail'], 1),
                           unguarded_aggr_breach=round(e['unguarded_aggr_breach'], 3),
                           fire_c5=bool(man.get('fire_c5', {}).get(str(k), False)),
                           held=bool(e['aggr_breach'] <= man['eps_target']),
                           sig=bool(e['p_sym'] <= 0.05)))
PANEL5 = pd.DataFrame(prows5)
if len(PANEL5):
    print(PANEL5.to_string(index=False))
    nh, ns, nt = int(PANEL5['held'].sum()), int(PANEL5['sig'].sum()), len(PANEL5)
    print(f'\n[common c] compliant cells: {nh}/{nt} | significant Δ(guarded−guarded), p≤0.05: {ns}/{nt} | '
          f'ties: {nt - ns}')
    out_csv = f'{DRIVE_REPO}/checkpoints/controls_panel_summary_{_c5tag5}.csv'
    PANEL5.to_csv(out_csv, index=False)
    with open(f'{DRIVE_REPO}/checkpoints/controls_panel_manifest_{_c5tag5}.json', 'w') as fh:
        _jj.dump({'created': _tt.strftime('%Y-%m-%d %H:%M:%S'), 'budget_mode': 'common',
                  'c_common_bp': (C_COMMON if 'C_COMMON' in globals() else 5.0),
                  'eps_target': 0.10,
                  'regimes_present': sorted(PANEL5['regime'].unique().tolist()),
                  'n_cells': nt, 'n_held': nh, 'n_significant': ns,
                  'note': 'unified-budget evaluation (supervisor request on §7.6): both arms guarded '
                          'by crc_rwc_aci at a common c; per-policy certification numbers remain in '
                          'controls_panel_summary.csv (v1.1) — the two conventions are complementary.'},
                 fh, ensure_ascii=False, indent=1)
    print('saved →', out_csv, f'+ controls_panel_manifest_{_c5tag5}.json')
else:
    print('nothing to aggregate yet.')